# ITI113 - Set Up SageMaker MLflow App

This notebook creates or reuses a **SageMaker MLflow App** for experiment tracking, then performs a small test MLflow run.

The notebook is now team-parameterised. Update only the configuration cell:

- `TEAM_ID`
- `STUDENT_ID`
- `PROJECT_NAME`

The MLflow App is created with tags such as `Course`, `TeamId`, and `ProjectName`.


## 1. Install or update required packages

The `sagemaker-mlflow` plugin lets the normal MLflow Python client authenticate to SageMaker MLflow using AWS IAM/SigV4. Restart the kernel after this cell if the notebook asks you to.

In [1]:
%pip install -q -U boto3 botocore mlflow sagemaker-mlflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
aiobotocore 3.8.0 requires botocore<1.43.47,>=1.43.3, but you have botocore 1.43.78 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.6.0 which is incompatible.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 which is incompatible.
Note: you may need to restart the ker

## 2. Configuration

Configure bucket, role and team naming.

The role used by the MLflow App must be able to access the S3 artifact store.


In [2]:
import boto3
from botocore.exceptions import ClientError
import time
import json
from datetime import datetime
from pathlib import Path

REGION = "ap-southeast-1"
COURSE = "ITI113"
SEMESTER = "26S1"

# Change these for each team/student.
TEAM_ID = "team07"
STUDENT_ID = "s701"

# Project name used in tags and artifact organisation.
PROJECT_NAME = "credit-card-fraud-detection"

# Existing course bucket from the SageMaker Pipeline lab.
CLASS_BUCKET = "nyp-26s1-iti113"

# One MLflow App per team is usually enough.
MLFLOW_APP_NAME = f"iti113-26s1-{TEAM_ID}-mlflow-app"

# Where MLflow run artifacts will be stored.
ARTIFACT_STORE_URI = f"s3://{CLASS_BUCKET}/iti113/{TEAM_ID}/mlflow-app-artifacts/"

# Experiment name inside MLflow App. Use a normal MLflow experiment name, not a Databricks /Workspace path.
EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/Experiment1"

# Optional: make this MLflow App the account/domain default. Keep False for classroom safety.
SET_AS_ACCOUNT_DEFAULT = False
SET_AS_DEFAULT_FOR_EXISTING_DOMAINS = False

session = boto3.Session(region_name=REGION)
sts = session.client("sts")
sm = session.client("sagemaker")
s3 = session.client("s3")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

# Build role name from TEAM_ID, e.g. team40 -> SageMakerExecutionRole-ITI113-Team07.
TEAM_ROLE_SUFFIX = TEAM_ID.lower().replace("team", "Team")
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/SageMakerExecutionRole-ITI113-{TEAM_ROLE_SUFFIX}"

# Required tags for team-level MLflow IAM restriction.
MLFLOW_APP_TAGS = [
    {"Key": "Course", "Value": COURSE},
    {"Key": "Semester", "Value": SEMESTER},
    {"Key": "TeamId", "Value": TEAM_ID},
    {"Key": "StudentId", "Value": STUDENT_ID},
    {"Key": "ProjectName", "Value": PROJECT_NAME},
    {"Key": "CreatedByNotebook", "Value": "Team07 MLOps Member"},
]

print("Account:", ACCOUNT_ID)
print("Caller ARN:", CALLER_ARN)
print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("Project Name:", PROJECT_NAME)
print("MLflow App Name:", MLFLOW_APP_NAME)
print("Artifact Store:", ARTIFACT_STORE_URI)
print("Role ARN:", ROLE_ARN)
print("Experiment:", EXPERIMENT_NAME)
print("\nMLflow App tags to apply:")
for tag in MLFLOW_APP_TAGS:
    print(f"  {tag['Key']} = {tag['Value']}")


Account: 044528205969
Caller ARN: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team07/SageMaker
Region: ap-southeast-1
Team ID: team07
Student ID: s701
Project Name: credit-card-fraud-detection
MLflow App Name: iti113-26s1-team07-mlflow-app
Artifact Store: s3://nyp-26s1-iti113/iti113/team07/mlflow-app-artifacts/
Role ARN: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team07
Experiment: ITI113/team07/Experiment1

MLflow App tags to apply:
  Course = ITI113
  Semester = 26S1
  TeamId = team07
  StudentId = s701
  ProjectName = credit-card-fraud-detection
  CreatedByNotebook = Team07 MLOps Member


## 3. Check SDK/API support

If this cell says `create_mlflow_app` is missing, update the Studio environment's `boto3` and `botocore`, then restart the kernel and rerun the notebook.


In [3]:
required_methods = [
    "create_mlflow_app",
    "list_mlflow_apps",
    "describe_mlflow_app",
    "create_presigned_mlflow_app_url",
    "add_tags",
    "list_tags",
]

missing = [method for method in required_methods if not hasattr(sm, method)]

print("SageMaker client supports:")
for method in required_methods:
    print(f"  {method}: {hasattr(sm, method)}")

if missing:
    raise RuntimeError(
        "Your boto3/botocore version does not support these SageMaker MLflow App/tag APIs: "
        + ", ".join(missing)
        + "\nRun the package update cell, restart the kernel, and rerun."
    )


SageMaker client supports:
  create_mlflow_app: True
  list_mlflow_apps: True
  describe_mlflow_app: True
  create_presigned_mlflow_app_url: True
  add_tags: True
  list_tags: True


## 4. Verify the S3 artifact store prefix is writable

This checks that the current role can write a small test file to the MLflow artifact prefix.


In [4]:
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

def parse_s3_uri(uri: str):
    parsed = urlparse(uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Not an S3 URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")

artifact_bucket, artifact_prefix = parse_s3_uri(ARTIFACT_STORE_URI)
test_key = f"{artifact_prefix.rstrip('/')}/_setup_test/{TEAM_ID}_{STUDENT_ID}_write_test.txt"
print(artifact_bucket)
print(artifact_prefix)
try:
    s3.put_object(
        Bucket=artifact_bucket,
        Key=test_key,
        Body=(
            f"MLflow App setup write test at {datetime.now(ZoneInfo("Asia/Singapore")).isoformat()}Z\n"
        ).encode("utf-8"),
    )
    print("S3 write test succeeded:")
    print(f"s3://{artifact_bucket}/{test_key}")
except ClientError:
    print("S3 write test failed. Check bucket/prefix permissions.")
    raise


nyp-26s1-iti113
iti113/team07/mlflow-app-artifacts/
S3 write test succeeded:
s3://nyp-26s1-iti113/iti113/team07/mlflow-app-artifacts/_setup_test/team07_s701_write_test.txt


## 5. Create or reuse the SageMaker MLflow App

This cell first checks whether an MLflow App with the configured name already exists.

- If the App does **not** exist, it creates the App with the required tags.
- If the App already exists, it reuses the App and checks whether the required tags are present.


In [5]:
def find_mlflow_app_by_name(name: str):
    paginator = sm.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        for summary in page.get("Summaries", []):
            if summary.get("Name") == name:
                return summary
    return None


def ensure_mlflow_app_tags(resource_arn: str, required_tags: list):
    """Check and apply required tags to an existing MLflow App.

    If the current role does not have sagemaker:AddTags/ListTags permission,
    this function will print a warning and continue. The admin can tag the App later.
    """
    required = {tag["Key"]: tag["Value"] for tag in required_tags}

    try:
        existing_tags_response = sm.list_tags(ResourceArn=resource_arn)
        existing = {
            tag["Key"]: tag["Value"]
            for tag in existing_tags_response.get("Tags", [])
        }

        missing_or_different = [
            {"Key": key, "Value": value}
            for key, value in required.items()
            if existing.get(key) != value
        ]

        if missing_or_different:
            print("\nAdding/updating required MLflow App tags:")
            for tag in missing_or_different:
                print(f"  {tag['Key']} = {tag['Value']}")

            sm.add_tags(
                ResourceArn=resource_arn,
                Tags=missing_or_different,
            )
        else:
            print("\nExisting MLflow App already has the required tags.")

        final_tags = sm.list_tags(ResourceArn=resource_arn).get("Tags", [])
        print("\nCurrent MLflow App tags:")
        for tag in final_tags:
            print(f"  {tag['Key']} = {tag['Value']}")

    except ClientError as e:
        print("\n[WARNING] Could not verify or update MLflow App tags.")
        print("This may happen if the current role does not have sagemaker:ListTags/AddTags.")
        print("Ask the admin to ensure these tags exist on the MLflow App:")
        for tag in required_tags:
            print(f"  {tag['Key']} = {tag['Value']}")
        print("\nOriginal error:")
        print(e)


existing = find_mlflow_app_by_name(MLFLOW_APP_NAME)

if existing:
    mlflow_app_arn = existing["Arn"]
    print("Reusing existing MLflow App:")
    print(json.dumps(existing, indent=2, default=str))

    # Important for team-level MLflow IAM restriction.
    ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)

else:
    create_args = {
        "Name": MLFLOW_APP_NAME,
        "ArtifactStoreUri": ARTIFACT_STORE_URI,
        "RoleArn": ROLE_ARN,
        "ModelRegistrationMode": "AutoModelRegistrationDisabled",
        "Tags": MLFLOW_APP_TAGS,
    }

    if SET_AS_ACCOUNT_DEFAULT:
        create_args["AccountDefaultStatus"] = "ENABLED"

    if SET_AS_DEFAULT_FOR_EXISTING_DOMAINS and domain_ids:
        create_args["DefaultDomainIdList"] = domain_ids

    print("Creating MLflow App with args:")
    print(json.dumps(create_args, indent=2, default=str))

    response = sm.create_mlflow_app(**create_args)
    mlflow_app_arn = response["Arn"]
    print("Create response:", response)

print("\nMLflow App ARN:")
print(mlflow_app_arn)

Reusing existing MLflow App:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O",
  "Name": "iti113-26s1-team07-mlflow-app",
  "Status": "Created",
  "CreationTime": "2026-07-24 06:41:43+00:00",
  "LastModifiedTime": "2026-08-04 14:14:18.899000+00:00",
  "MlflowVersion": "3.10.1"
}

Existing MLflow App already has the required tags.

Current MLflow App tags:
  Semester = 26S1
  sagemaker:domain-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-cydcdxkc4yot
  ProjectName = credit-card-fraud-detection
  sagemaker:space-arn = arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-cydcdxkc4yot/team07-shared
  Course = ITI113
  TeamId = team07
  CreatedByNotebook = Team07 MLOps Member
  StudentId = s701

MLflow App ARN:
arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O


In [6]:
def wait_for_mlflow_app(arn: str, timeout_seconds: int = 600, poll_seconds: int = 20):
    start = time.time()
    last_status = None

    while True:
        desc = sm.describe_mlflow_app(Arn=arn)
        status = desc.get("Status")

        if status != last_status:
            print(f"Status: {status}")
            last_status = status

        if status in ["Created", "Updated"]:
            return desc

        if status in ["CreateFailed", "UpdateFailed", "DeleteFailed", "Deleted"]:
            raise RuntimeError(
                f"MLflow App entered failure status: {status}\n"
                + json.dumps(desc, indent=2, default=str)
            )

        if time.time() - start > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for MLflow App. Last status: {status}")

        time.sleep(poll_seconds)

mlflow_app_desc = wait_for_mlflow_app(mlflow_app_arn)

print("\nFinal MLflow App description:")
print(json.dumps(mlflow_app_desc, indent=2, default=str))


Status: Created

Final MLflow App description:
{
  "Arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O",
  "Name": "iti113-26s1-team07-mlflow-app",
  "ArtifactStoreUri": "s3://nyp-26s1-iti113/iti113/team07/mlflow-app-artifacts/",
  "MlflowVersion": "3.10.1",
  "RoleArn": "arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team07",
  "Status": "Created",
  "ModelRegistrationMode": "AutoModelRegistrationDisabled",
  "CreationTime": "2026-07-24 06:41:43+00:00",
  "CreatedBy": {
    "IamIdentity": {
      "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team07/SageMaker"
    }
  },
  "LastModifiedTime": "2026-08-04 14:14:18.899000+00:00",
  "LastModifiedBy": {
    "IamIdentity": {
      "Arn": "arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team07/SageMaker"
    }
  },
  "WeeklyMaintenanceWindowStart": "Tue:14:43",
  "MaintenanceStatus": "MAINTENANCE_COMPLETE",
  "ResponseMetadata": {
    "RequestId":

## 6. Get a presigned MLflow UI URL

The URL is usually single-use and expires. Generate a fresh URL whenever you want to open the MLflow UI.


In [11]:
url_response = sm.create_presigned_mlflow_app_url(
    Arn=mlflow_app_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=3600,
)

mlflow_ui_url = url_response["AuthorizedUrl"]

print("Open this MLflow UI URL in a browser tab:")
print(mlflow_ui_url)

Open this MLflow UI URL in a browser tab:
https://app-PVSI6X27672O.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlBXWVZVUSIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHBBSjUrd1BGSFVuTWc2K0JieFdRVGhRYk9KK0NxeGNBMVVkOVFEdHdpN29BWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFeVZuWjNWVU0wUmxwTmNVVTBSMllySzBWSlJrOTBabUpDV1hsWWRHcEhSM1ZVVUdWMmNpdHBkVFZoZGxWUmNuY3JXVTVEVDJ3MFJETkVLMmN6UWpSaVVUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFUWlNaNnhUOTIzYkh0QkplWmUwdC80QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4b0VJc3pCU2dCbm5CMXFOWUNBUkNBTy81SUw2TFVZWS9qcmtEZjc2UVJyNTJkMmJEanRDc3QwZ3lnUnYxamE4dStXRCt1am5tNlNMamlacGZWZkYvKzhWUG9HQlRaUWhQL01SVFpBZ0FBRUFCNXNOTHMxZFMrU1Z6Y3NjL2tZYjMwczhTeHVLNmFEa0FTYTd5Tk91dkIrNmd0c

## 7. Test MLflow logging against the SageMaker MLflow App

This uses the SageMaker MLflow App ARN as the MLflow tracking URI. No Databricks host or token is required.


In [8]:
import mlflow
import tempfile
from pathlib import Path

print("MLflow version:", mlflow.__version__)

mlflow.set_tracking_uri(mlflow_app_arn)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"{TEAM_ID}_{STUDENT_ID}_mlflow_app_test"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({
        "course": COURSE,
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "tracking_backend": "sagemaker_mlflow_app",
        "purpose": "setup_validation",
        "project_name": PROJECT_NAME,
    })

    mlflow.log_params({
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "region": REGION,
        "artifact_store_uri": ARTIFACT_STORE_URI,
        "project_name": PROJECT_NAME,
    })

    mlflow.log_metrics({
        "test_accuracy": 0.888,
        "test_f1": 0.876,
        "test_auc_roc": 0.901,
    })

    summary = {
        "message": "SageMaker MLflow App logging test succeeded.",
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
        "mlflow_app_arn": mlflow_app_arn,
        "experiment_name": EXPERIMENT_NAME,
        "run_id": run.info.run_id,
    }

    with tempfile.TemporaryDirectory() as tmpdir:
        artifact_path = Path(tmpdir) / "mlflow_app_setup_summary.json"
        artifact_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(artifact_path), artifact_path="setup_test")

    run_id = run.info.run_id

print("Logged test run successfully.")
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", run_name)
print("Run ID:", run_id)

MLflow version: 3.15.1
🏃 View run team07_s701_mlflow_app_test at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/c96a7ee90c764928af1ddf4e16b462ca
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
Logged test run successfully.
Experiment: ITI113/team07/Experiment1
Run name: team07_s701_mlflow_app_test
Run ID: c96a7ee90c764928af1ddf4e16b462ca


## 8. Search the test run

This confirms that the run is visible through the MLflow API.


In [9]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError(f"Experiment not found: {EXPERIMENT_NAME}")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.student_id = '{STUDENT_ID}' and tags.team_id = '{TEAM_ID}'",
    order_by=["attributes.start_time DESC"],
    max_results=10,
)

runs[["run_id", "tags.mlflow.runName", "metrics.test_accuracy", "metrics.test_f1", "metrics.test_auc_roc"]]


,run_id,tags.mlflow.runName,metrics.test_accuracy,metrics.test_f1,metrics.test_auc_roc
0,c96a7ee90c764928af1ddf4e16b462ca,team07_s701_mlflow_app_test,0.8880,0.876000,0.901000
1,c2a31c1cc2b84ba2b8a5b5c6ed369226,team07_s701_pipeline_XGBoost_1787553892,0.8562,0.739964,0.813436
2,55f4b1f61f774bc09a33ecfadd98fc65,team07_s701_pipeline_precheck_1787553146,NaN,NaN,NaN
3,bd6bda9214c347a4b0dbcf72c2176e0d,xgboost_candidate_06,NaN,NaN,NaN
4,b5b7943b5feb4629a5f1810ff572b623,xgboost_candidate_05,0.8561,0.739830,0.814480
5,cd6b719e68ac481d91f908aa688108dd,xgboost_candidate_04,NaN,NaN,NaN
6,0e217a919afb44c58f1e09b442d0498b,xgboost_candidate_03,NaN,NaN,NaN
7,887a46713b1f4b769785e3f6abda1cc0,xgboost_candidate_02,NaN,NaN,NaN
8,a25e5633a944469980cb93cd18448c4a,xgboost_candidate_01,NaN,NaN,NaN
9,6c84f1bf9378422ab0f4f94f1e17ceaa,team07_s701_model_b_xgboost_tuning,NaN,NaN,NaN


## 9. Save Shared MLflow Configuration for Expriments and Pipeline

Save the AWS, MLflow, team, and project settings required by subsequent experiment and pipeline notebooks. Later notebooks can load the generated JSON configuration file, avoiding manual copying and inconsistent settings.

In [10]:
print("# Shared configuration for subsequent notebooks")
print(f'REGION = "{REGION}"')
print(f'MLFLOW_APP_ARN = "{mlflow_app_arn}"')
print(f'EXPERIMENT_NAME = "{EXPERIMENT_NAME}"')
print(f'TEAM_ID = "{TEAM_ID}"')
print(f'STUDENT_ID = "{STUDENT_ID}"')
print(f'PROJECT_NAME = "{PROJECT_NAME}"')

# Save locally for convenience.
config = {
    "REGION": REGION,
    "MLFLOW_APP_ARN": mlflow_app_arn,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "TEAM_ID": TEAM_ID,
    "STUDENT_ID": STUDENT_ID,
    "PROJECT_NAME": PROJECT_NAME,
    "ARTIFACT_STORE_URI": ARTIFACT_STORE_URI,
    "MLFLOW_APP_NAME": MLFLOW_APP_NAME,
    "MLFLOW_APP_TAGS": MLFLOW_APP_TAGS,
}

savepath = f"mlflow_app_config_{TEAM_ID.lower()}_{STUDENT_ID.lower()}.json"

Path(savepath).write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print(f"\nConfiguration saved successfully: {savepath}")
print("Use this JSON file in subsequent experiment and pipeline notebooks.")

# Shared configuration for subsequent notebooks
REGION = "ap-southeast-1"
MLFLOW_APP_ARN = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PVSI6X27672O"
EXPERIMENT_NAME = "ITI113/team07/Experiment1"
TEAM_ID = "team07"
STUDENT_ID = "s701"
PROJECT_NAME = "credit-card-fraud-detection"

Configuration saved successfully: mlflow_app_config_team07_s701.json
Use this JSON file in subsequent experiment and pipeline notebooks.
